## Carga de los datos

In [4]:
from pathlib import Path

import pandas as pd
import numpy as np

# Identificar raíz del proyecto
cwd = Path.cwd().resolve()

PROJECT_ROOT = next(
    (
        ruta
        for ruta in (cwd, *cwd.parents)
        if (ruta / "data" / "interim").is_dir()
        and (ruta / "notebooks").is_dir()
    ),
    None
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "No se encontró la raíz del proyecto."
    )

# Directorios
INTERIM_DIR = PROJECT_ROOT / "data" / "interim"

# Cargar información
panel = pd.read_parquet(
    INTERIM_DIR / "panel_bancario_2026_09.parquet"
)

# Normalizar fechas
panel["fecha"] = pd.to_datetime(panel["fecha"])

# Ordenar
panel = (
    panel
    .sort_values(["banco", "fecha"])
    .reset_index(drop=True)
)

# Comprobar integridad
assert not panel.duplicated(["banco", "fecha"]).any()

print("Dimensiones:", panel.shape)
print("Instituciones:", panel["banco"].nunique())

display(panel.head())

Dimensiones: (5972, 11)
Instituciones: 23


,fecha,banco,solvencia,liquidez,roa,roe,cartera_depositos,cartera_bruta,cartera_improductiva,morosidad_total,cobertura
0,2011-07-01,BANCO ATLANTIDA S.A.,NaN,0.0,0.019541,0.091531,0.0,31872.72431,828.74366,0.026002,0.776931
1,2011-08-01,BANCO ATLANTIDA S.A.,0.25177,0.0,0.018258,0.085764,0.0,33162.49816,728.98974,0.021982,1.002009
2,2011-09-01,BANCO ATLANTIDA S.A.,0.28814,0.0,0.017008,0.081672,0.0,34827.65249,822.65943,0.023621,1.003828
3,2011-10-01,BANCO ATLANTIDA S.A.,0.26950,0.0,0.016025,0.073058,0.0,36890.47681,909.87157,0.024664,1.026205
4,2011-11-01,BANCO ATLANTIDA S.A.,0.26057,0.0,0.015737,0.070117,0.0,38955.55403,1020.61617,0.026200,1.007305


## Construccion del target de los 3 meses siguientes

In [5]:
# ==========================================
# CONSTRUCCIÓN DE TARGETS MULTIHORIZONTE
# ==========================================

HORIZONTES = [1, 2, 3]

# Partir del panel histórico completo
df_features = (
    panel
    .sort_values(["banco", "fecha"])
    .reset_index(drop=True)
    .copy()
)

# Convertir morosidad a puntos porcentuales
df_features["morosidad_pp"] = (
    df_features["morosidad_total"] * 100
)

# Verificar integridad
assert not df_features.duplicated(
    ["banco", "fecha"]
).any()

assert df_features["fecha"].dt.is_month_start.all()


# Construir un target por horizonte
for h in HORIZONTES:

    nombre_target = f"target_morosidad_t{h}_pp"

    # Tabla auxiliar con morosidad futura
    futuro = df_features[
        ["banco", "fecha", "morosidad_pp"]
    ].copy()

    # Desplazar las fechas h meses hacia atrás
    futuro["fecha"] = (
        futuro["fecha"]
        - pd.DateOffset(months=h)
    )

    # Renombrar variable objetivo
    futuro = futuro.rename(
        columns={"morosidad_pp": nombre_target}
    )

    # Incorporar target mediante banco y fecha
    df_features = df_features.merge(
        futuro,
        on=["banco", "fecha"],
        how="left",
        validate="one_to_one"
    )

    # Fecha calendario correspondiente al objetivo
    df_features[f"fecha_objetivo_t{h}"] = (
        df_features["fecha"]
        + pd.DateOffset(months=h)
    )


# Visualizar resultados
columnas = [
    "fecha",
    "banco",
    "morosidad_pp",
    "target_morosidad_t1_pp",
    "target_morosidad_t2_pp",
    "target_morosidad_t3_pp"
]

display(df_features[columnas].head(10))

,fecha,banco,morosidad_pp,target_morosidad_t1_pp,target_morosidad_t2_pp,target_morosidad_t3_pp
0,2011-07-01,BANCO ATLANTIDA S.A.,2.600166,2.198235,2.362087,2.466413
1,2011-08-01,BANCO ATLANTIDA S.A.,2.198235,2.362087,2.466413,2.619950
2,2011-09-01,BANCO ATLANTIDA S.A.,2.362087,2.466413,2.619950,2.230437
3,2011-10-01,BANCO ATLANTIDA S.A.,2.466413,2.619950,2.230437,2.682136
4,2011-11-01,BANCO ATLANTIDA S.A.,2.619950,2.230437,2.682136,3.232366
5,2011-12-01,BANCO ATLANTIDA S.A.,2.230437,2.682136,3.232366,2.989239
6,2012-01-01,BANCO ATLANTIDA S.A.,2.682136,3.232366,2.989239,3.347692
7,2012-02-01,BANCO ATLANTIDA S.A.,3.232366,2.989239,3.347692,3.623611
8,2012-03-01,BANCO ATLANTIDA S.A.,2.989239,3.347692,3.623611,4.045769
9,2012-04-01,BANCO ATLANTIDA S.A.,3.347692,3.623611,4.045769,3.914882


## Comprobar las fronteras temporales

In [7]:
# ==========================================
# CARACTERÍSTICAS TEMPORALES
# ==========================================

features_temporales = df_features.copy()

# Rezagos históricos
REZAGOS = [1, 2, 3, 6, 12]

for k in REZAGOS:

    # Tabla auxiliar
    historico = df_features[
        ["banco", "fecha", "morosidad_pp"]
    ].copy()

    # Desplazar fechas hacia adelante
    historico["fecha"] = (
        historico["fecha"]
        + pd.DateOffset(months=k)
    )

    # Nombre de la característica
    nombre_lag = f"morosidad_lag_{k}_pp"

    historico = historico.rename(
        columns={"morosidad_pp": nombre_lag}
    )

    # Incorporar rezago
    features_temporales = features_temporales.merge(
        historico,
        on=["banco", "fecha"],
        how="left",
        validate="one_to_one"
    )


# ==========================================
# VARIACIONES HISTÓRICAS
# ==========================================

# Variación del último mes
features_temporales["delta_morosidad_1m_pp"] = (
    features_temporales["morosidad_pp"]
    - features_temporales["morosidad_lag_1_pp"]
)

# Variación acumulada de tres meses
features_temporales["delta_morosidad_3m_pp"] = (
    features_temporales["morosidad_pp"]
    - features_temporales["morosidad_lag_3_pp"]
)


# Visualizar
display(
    features_temporales[
        [
            "fecha",
            "banco",
            "morosidad_pp",
            "morosidad_lag_1_pp",
            "morosidad_lag_3_pp",
            "delta_morosidad_1m_pp",
            "delta_morosidad_3m_pp",
            "target_morosidad_t1_pp",
            "target_morosidad_t2_pp",
            "target_morosidad_t3_pp"
        ]
    ].head(15)
)

,fecha,banco,morosidad_pp,morosidad_lag_1_pp,morosidad_lag_3_pp,delta_morosidad_1m_pp,delta_morosidad_3m_pp,target_morosidad_t1_pp,target_morosidad_t2_pp,target_morosidad_t3_pp
0,2011-07-01,BANCO ATLANTIDA S.A.,2.600166,NaN,NaN,NaN,NaN,2.198235,2.362087,2.466413
1,2011-08-01,BANCO ATLANTIDA S.A.,2.198235,2.600166,NaN,-0.401930,NaN,2.362087,2.466413,2.619950
2,2011-09-01,BANCO ATLANTIDA S.A.,2.362087,2.198235,NaN,0.163852,NaN,2.466413,2.619950,2.230437
3,2011-10-01,BANCO ATLANTIDA S.A.,2.466413,2.362087,2.600166,0.104326,-0.133753,2.619950,2.230437,2.682136
4,2011-11-01,BANCO ATLANTIDA S.A.,2.619950,2.466413,2.198235,0.153537,0.421715,2.230437,2.682136,3.232366
5,2011-12-01,BANCO ATLANTIDA S.A.,2.230437,2.619950,2.362087,-0.389514,-0.131650,2.682136,3.232366,2.989239
6,2012-01-01,BANCO ATLANTIDA S.A.,2.682136,2.230437,2.466413,0.451699,0.215723,3.232366,2.989239,3.347692
7,2012-02-01,BANCO ATLANTIDA S.A.,3.232366,2.682136,2.619950,0.550230,0.612416,2.989239,3.347692,3.623611
8,2012-03-01,BANCO ATLANTIDA S.A.,2.989239,3.232366,2.230437,-0.243127,0.758803,3.347692,3.623611,4.045769
9,2012-04-01,BANCO ATLANTIDA S.A.,3.347692,2.989239,2.682136,0.358453,0.665556,3.623611,4.045769,3.914882


## Incorporar estacionalidad

In [8]:
# ==========================================
# CARACTERÍSTICAS ESTACIONALES MULTIHORIZONTE
# ==========================================

for h in HORIZONTES:

    # Fecha objetivo
    fecha_objetivo = (
        features_temporales[f"fecha_objetivo_t{h}"]
    )

    # Mes del objetivo
    mes = fecha_objetivo.dt.month

    # Guardar mes
    features_temporales[f"mes_objetivo_t{h}"] = mes

    # Codificación cíclica
    features_temporales[f"mes_sin_t{h}"] = np.sin(
        2 * np.pi * mes / 12
    )

    features_temporales[f"mes_cos_t{h}"] = np.cos(
        2 * np.pi * mes / 12
    )


# Comprobar un ejemplo
display(
    features_temporales.loc[
        (
            features_temporales["banco"] == "BP PICHINCHA"
        )
        & (
            features_temporales["fecha"]
            == "2025-09-01"
        ),
        [
            "fecha",
            "fecha_objetivo_t1",
            "fecha_objetivo_t2",
            "fecha_objetivo_t3",
            "mes_objetivo_t1",
            "mes_objetivo_t2",
            "mes_objetivo_t3",
            "mes_sin_t1",
            "mes_cos_t1",
            "mes_sin_t2",
            "mes_cos_t2",
            "mes_sin_t3",
            "mes_cos_t3"
        ]
    ]
)

,fecha,fecha_objetivo_t1,fecha_objetivo_t2,fecha_objetivo_t3,mes_objetivo_t1,mes_objetivo_t2,mes_objetivo_t3,mes_sin_t1,mes_cos_t1,mes_sin_t2,mes_cos_t2,mes_sin_t3,mes_cos_t3
4983,2025-09-01,2025-10-01,2025-11-01,2025-12-01,10,11,12,-0.866025,0.5,-0.5,0.866025,-2.449294e-16,1.0


## Validar los targets

In [9]:
# ==========================================
# VALIDACIÓN MULTIHORIZONTE
# ==========================================

print("DIMENSIONES")
print(features_temporales.shape)

print("\nINSTITUCIONES")
print(features_temporales["banco"].nunique())

print("\nTARGETS POR HORIZONTE")

for h in HORIZONTES:

    columna = f"target_morosidad_t{h}_pp"
    fecha_columna = f"fecha_objetivo_t{h}"

    disponibles = features_temporales[columna].notna().sum()
    faltantes = features_temporales[columna].isna().sum()

    # Verificar fechas objetivo
    fechas_esperadas = (
        features_temporales["fecha"]
        + pd.DateOffset(months=h)
    )

    assert (
        features_temporales[fecha_columna]
        == fechas_esperadas
    ).all()

    print(f"\nHorizonte: {h} mes(es)")
    print("Targets disponibles:", disponibles)
    print("Targets faltantes:", faltantes)


# Validaciones estructurales
assert len(features_temporales) == len(panel)

assert not features_temporales.duplicated(
    ["banco", "fecha"]
).any()

assert (
    features_temporales["target_morosidad_t1_pp"]
    .notna()
    .sum()
    == 5944
)

print("\nVALIDACIÓN ESTRUCTURAL COMPLETADA")

DIMENSIONES
(5972, 34)

INSTITUCIONES
23

TARGETS POR HORIZONTE

Horizonte: 1 mes(es)
Targets disponibles: 5944
Targets faltantes: 28

Horizonte: 2 mes(es)
Targets disponibles: 5922
Targets faltantes: 50

Horizonte: 3 mes(es)
Targets disponibles: 5900
Targets faltantes: 72

VALIDACIÓN ESTRUCTURAL COMPLETADA


## Separacion entre desarrollo y test

In [10]:
# ==========================================
# SEPARACIÓN TEMPORAL MULTIHORIZONTE
# ==========================================

TEST_INICIO = pd.Timestamp("2025-10-01")
TEST_FIN = pd.Timestamp("2026-09-01")

# Último mes utilizado para entrenar
CORTE_ENTRENAMIENTO = pd.Timestamp("2025-09-01")

# Almacenar datasets
datasets_horizontes = {}

for h in HORIZONTES:

    target = f"target_morosidad_t{h}_pp"
    fecha_target = f"fecha_objetivo_t{h}"

    # Seleccionar observaciones con target válido
    datos = features_temporales.dropna(
        subset=[target]
    ).copy()

    # Entrenamiento/desarrollo:
    # solo objetivos anteriores al período Test
    desarrollo_h = datos.loc[
        datos[fecha_target] < TEST_INICIO
    ].copy()

    # Test:
    # objetivos dentro del período reservado
    # y pronósticos emitidos desde el corte
    test_h = datos.loc[
        datos[fecha_target].between(
            TEST_INICIO,
            TEST_FIN
        )
        & (datos["fecha"] >= CORTE_ENTRENAMIENTO)
    ].copy()

    # Verificar frontera temporal
    assert (
        desarrollo_h[fecha_target].max()
        <= CORTE_ENTRENAMIENTO
    )

    assert (
        test_h["fecha"].min()
        >= CORTE_ENTRENAMIENTO
    )

    # Guardar
    datasets_horizontes[h] = {
        "desarrollo": desarrollo_h,
        "test": test_h
    }

    print(f"\nHORIZONTE {h} MES(ES)")
    print("Desarrollo:", len(desarrollo_h))
    print("Test:", len(test_h))

    print(
        "Primer target Test:",
        test_h[fecha_target].min()
    )

    print(
        "Último target Test:",
        test_h[fecha_target].max()
    )


HORIZONTE 1 MES(ES)
Desarrollo: 5668
Test: 276
Primer target Test: 2025-10-01 00:00:00
Último target Test: 2026-09-01 00:00:00

HORIZONTE 2 MES(ES)
Desarrollo: 5646
Test: 253
Primer target Test: 2025-11-01 00:00:00
Último target Test: 2026-09-01 00:00:00

HORIZONTE 3 MES(ES)
Desarrollo: 5624
Test: 230
Primer target Test: 2025-12-01 00:00:00
Último target Test: 2026-09-01 00:00:00


## Estadisticas moviles de morosidad

### Construccion de las estadisticas moviles

In [11]:
# ==========================================
# ESTADÍSTICAS MÓVILES DE MOROSIDAD
# ==========================================

VENTANAS = [3, 6]

resultados_moviles = []

for banco, datos in features_temporales.groupby("banco"):

    # Construir serie mensual por institución
    serie = (
        datos
        .set_index("fecha")["morosidad_pp"]
        .sort_index()
        .asfreq("MS")
    )

    # Tabla auxiliar
    auxiliar = pd.DataFrame({
        "banco": banco,
        "fecha": serie.index
    })

    for ventana in VENTANAS:

        # Ventana móvil
        movil = serie.rolling(
            window=ventana,
            min_periods=ventana
        )

        # Media móvil
        auxiliar[f"morosidad_media_{ventana}m_pp"] = (
            movil.mean().to_numpy()
        )

        # Desviación estándar móvil
        auxiliar[f"morosidad_std_{ventana}m_pp"] = (
            movil.std(ddof=0).to_numpy()
        )

    resultados_moviles.append(auxiliar)


# Unificar características
moviles = pd.concat(
    resultados_moviles,
    ignore_index=True
)

# Incorporarlas al dataset
features_temporales = features_temporales.merge(
    moviles,
    on=["banco", "fecha"],
    how="left",
    validate="one_to_one"
)

# Comprobar resultados
columnas_moviles = [
    "morosidad_media_3m_pp",
    "morosidad_std_3m_pp",
    "morosidad_media_6m_pp",
    "morosidad_std_6m_pp"
]

print("Dimensiones:", features_temporales.shape)

print("\nValores faltantes:")
display(
    features_temporales[columnas_moviles]
    .isna()
    .sum()
)

display(
    features_temporales[
        ["fecha", "banco", "morosidad_pp"]
        + columnas_moviles
    ].head(15)
)

Dimensiones: (5972, 38)

Valores faltantes:


morosidad_media_3m_pp     53
morosidad_std_3m_pp       53
morosidad_media_6m_pp    122
morosidad_std_6m_pp      122
dtype: int64

,fecha,banco,morosidad_pp,morosidad_media_3m_pp,morosidad_std_3m_pp,morosidad_media_6m_pp,morosidad_std_6m_pp
0,2011-07-01,BANCO ATLANTIDA S.A.,2.600166,NaN,NaN,NaN,NaN
1,2011-08-01,BANCO ATLANTIDA S.A.,2.198235,NaN,NaN,NaN,NaN
2,2011-09-01,BANCO ATLANTIDA S.A.,2.362087,2.386829,0.165017,NaN,NaN
3,2011-10-01,BANCO ATLANTIDA S.A.,2.466413,2.342245,0.110378,NaN,NaN
4,2011-11-01,BANCO ATLANTIDA S.A.,2.619950,2.482817,0.105909,NaN,NaN
5,2011-12-01,BANCO ATLANTIDA S.A.,2.230437,2.438933,0.160201,2.412881,0.164701
6,2012-01-01,BANCO ATLANTIDA S.A.,2.682136,2.510841,0.199894,2.426543,0.182145
7,2012-02-01,BANCO ATLANTIDA S.A.,3.232366,2.714979,0.409695,2.598898,0.320949
8,2012-03-01,BANCO ATLANTIDA S.A.,2.989239,2.967914,0.225136,2.703424,0.328832
9,2012-04-01,BANCO ATLANTIDA S.A.,3.347692,3.189766,0.149406,2.850303,0.382589


## Ingenieria de caracteristicas financieras: Crecimiento de cartera

Construccion de caracteristicas:

1. Crecimiento mensual de cartera
2. Crecimiento trimestral de cartera